In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
%run ../00-common/05.data-quality-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df = (
  spark.table(bronze_table)
       .filter((F.col("batch_id") == v_batch_id))
       .select("season",
                "round",
                "constructorId",
                "driverId",
                "date",
                "raceName",
                "grid",
                "laps",
                "number",
                "points",
                "position",
                "positionText",
                "status",
                "ingestion_timestamp",
                "source_file",
                "batch_id")
       .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "final_position",
            "positionText": "final_position_text"
        })
)

In [0]:
results_valid_df = (
    results_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull() 
        )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)

In [0]:
results_final_df = (
    results_valid_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)

In [0]:
dq_checks = [
    # not_null
    {"name": "season_not_null",         "type": "not_null", "column": "season",         "severity": "critical"},
    {"name": "round_not_null",          "type": "not_null", "column": "round",          "severity": "critical"},
    {"name": "constructor_id_not_null", "type": "not_null", "column": "constructor_id", "severity": "critical"},
    {"name": "driver_id_not_null",      "type": "not_null", "column": "driver_id",      "severity": "critical"},
    {"name": "points_not_null",         "type": "not_null", "column": "points",         "severity": "warning"},
    {"name": "final_position_not_null", "type": "not_null", "column": "final_position", "severity": "warning"},
    {"name": "status_not_null",         "type": "not_null", "column": "status",         "severity": "warning"},
    # unique
    {"name": "composite_key_unique",
     "type": "unique",
     "columns": ["season", "round", "constructor_id", "driver_id"],
     "severity": "critical"},
    # min_rows
    {"name": "min_one_row", "type": "min_rows", "threshold": 1, "severity": "warning"},
    # value_range
    {"name": "points_not_negative",      "type": "not_negative",  "column": "points",        "severity": "warning"},
    {"name": "grid_position_positive",   "type": "value_range",   "column": "grid_position", "min": 0, "max": 30, "severity": "warning"},
    {"name": "completed_laps_not_negative", "type": "not_negative","column": "completed_laps","severity": "warning"},
    # reference integrity
    {
        "name": "driver_id_ref_drivers",
        "type": "referential", "column": "driver_id",
        "ref_table": f"{catalog_name}.{silver_schema}.drivers",
        "ref_column": "driver_id", "severity": "warning"
    },
    {
        "name": "constructor_id_ref_constructors",
        "type": "referential", "column": "constructor_id",
        "ref_table": f"{catalog_name}.{silver_schema}.constructors",
        "ref_column": "constructor_id", "severity": "warning"
    },
]

run_dq_checks(
    df                  = results_final_df, 
    checks              = dq_checks,
    table_name          = silver_table, 
    batch_id            = v_batch_id, 
    raise_on_critical   = True
)

In [0]:
apply_delta_constraints(
    table_name  = silver_table,
    constraints = [
        {"name": "chk_results_season_not_null",         "condition": "season IS NOT NULL"},
        {"name": "chk_results_round_not_null",          "condition": "round IS NOT NULL"},
        {"name": "chk_results_constructor_id_not_null", "condition": "constructor_id IS NOT NULL"},
        {"name": "chk_results_driver_id_not_null",      "condition": "driver_id IS NOT NULL"},
        {"name": "chk_results_points_not_negative",     "condition": "points IS NULL OR points >= 0"},
    ]
)

In [0]:
write_to_silver(
    input_df = results_final_df,
    target_table = silver_table,
    merge_condition = "t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
    columns_to_update = [
        "race_name",
        "race_date",
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)